In [1]:
import pandas as pd
import re

# google cloud library
from googleapiclient import discovery
from google.auth import default

/home/kelechi/miniconda3/envs/bio_ramp_env/lib/python3.9/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.21). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/home/kelechi/miniconda3/envs/bio_ramp_env/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/home/kelechi/miniconda3/envs/bio_ramp_env/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its e

In [2]:
# Load the NER-tagged dataset produced by all_dataset_ner.ipynb.
# This already contains norm_human_transcript_ner plus the raw normalized ASR columns.
df = pd.read_excel("results/all_result_processed_normalized_with_ner_tagged.xlsx", index_col=False)
print(f"Loaded {len(df)} rows")
print([c for c in df.columns if 'ner' in c.lower() or c.startswith('norm_')])

Loaded 120 rows
['norm_human_transcript', 'norm_whisper_asr', 'norm_phi4_asr', 'norm_parakeet_asr', 'norm_whisper_asr_wer', 'norm_whisper_asr_ins', 'norm_whisper_asr_del', 'norm_whisper_asr_sub', 'norm_phi4_asr_wer', 'norm_phi4_asr_ins', 'norm_phi4_asr_del', 'norm_phi4_asr_sub', 'norm_parakeet_asr_wer', 'norm_parakeet_asr_ins', 'norm_parakeet_asr_del', 'norm_parakeet_asr_sub', 'norm_whisper_asr_Deletions', 'norm_whisper_asr_Insertions', 'norm_whisper_asr_Substitutions', 'norm_phi4_asr_Deletions', 'norm_phi4_asr_Insertions', 'norm_phi4_asr_Substitutions', 'norm_parakeet_asr_Deletions', 'norm_parakeet_asr_Insertions', 'norm_parakeet_asr_Substitutions', 'norm_human_transcript_ner']


In [3]:
# NER helper functions (copied from all_dataset_ner.ipynb so this notebook is self-contained).
PROJECT_ID = "bio-ramp-ner"
LOCATION = "us-central1"


def analyze_medical_text(project_id, location, text_content):
    """
    Call Google Healthcare API to analyze medical entities in text.
    Returns full API response payload for downstream mention reconstruction.
    """
    try:
        credentials, _ = default()
        service = discovery.build("healthcare", "v1", credentials=credentials)

        nlp_service_name = f"projects/{project_id}/locations/{location}/services/nlp"
        body = {"documentContent": text_content}

        response = service.projects().locations().services().nlp().analyzeEntities(
            nlpService=nlp_service_name,
            body=body,
        ).execute()

        return response
    except Exception as e:
        print(f"Error calling healthcare API: {e}")
        return {}


def _collect_mentions_from_response(response, mention_types=None):
    """
    Extract mention spans in a normalized shape:
    {begin, end, replacement, type}
    """
    if not response:
        return []

    entities = response.get("entities", []) or []
    entity_mentions = response.get("entityMentions", []) or []

    mentions_to_replace = []

    if entity_mentions:
        for mention in entity_mentions:
            mention_type = mention.get("type")
            if mention_types and mention_type not in mention_types:
                continue

            text_obj = mention.get("text", {}) or {}
            surface_text = text_obj.get("content", "")
            begin_offset = text_obj.get("beginOffset")

            if begin_offset is None or not surface_text:
                continue

            end_offset = begin_offset + len(surface_text)
            replacement_tag = f"[{mention_type}: {surface_text}]"

            mentions_to_replace.append(
                {
                    "begin": begin_offset,
                    "end": end_offset,
                    "replacement": replacement_tag,
                    "type": mention_type,
                }
            )

        return mentions_to_replace

    for entity in entities:
        mentions = entity.get("mentions", []) or []
        for mention in mentions:
            mention_type = mention.get("type")
            if mention_types and mention_type not in mention_types:
                continue

            text_obj = mention.get("text", {}) or {}
            surface_text = text_obj.get("content", "")
            begin_offset = text_obj.get("beginOffset")

            if begin_offset is None or not surface_text:
                continue

            end_offset = begin_offset + len(surface_text)
            replacement_tag = f"[{mention_type}: {surface_text}]"
            mentions_to_replace.append(
                {
                    "begin": begin_offset,
                    "end": end_offset,
                    "replacement": replacement_tag,
                    "type": mention_type,
                }
            )

    return mentions_to_replace


def reconstruct_text_with_ner_tags(text_content, response, mention_types=None):
    """
    Reconstruct the original text with inline NER tags.
    Replaces mention spans with format: [MENTION_TYPE: TextSpan]
    """
    if pd.isna(text_content) or text_content is None:
        return text_content

    text_content = str(text_content)
    mentions_to_replace = _collect_mentions_from_response(response, mention_types=mention_types)
    if not mentions_to_replace:
        return text_content

    mentions_to_replace.sort(key=lambda x: x["begin"], reverse=True)

    reconstructed_text = text_content
    last_begin = len(text_content) + 1
    for mention in mentions_to_replace:
        begin = mention["begin"]
        end = mention["end"]
        replacement = mention["replacement"]

        if begin < 0 or end > len(reconstructed_text) or begin >= end:
            continue

        if end > last_begin:
            continue

        reconstructed_text = (
            reconstructed_text[:begin] + replacement + reconstructed_text[end:]
        )
        last_begin = begin

    return reconstructed_text


def apply_ner_to_row(text_content, project_id, location, mention_types=None):
    """Apply NER analysis to one transcript row and return the tagged transcript."""
    if pd.isna(text_content) or text_content is None:
        return text_content

    text_content = str(text_content)
    if not text_content.strip():
        return text_content

    response = analyze_medical_text(project_id, location, text_content)
    return reconstruct_text_with_ner_tags(text_content, response, mention_types=mention_types)


def extract_medical_entities_from_ner_tags(ner_tagged_text):
    """Extract medical entity spans from inline NER tags."""
    if pd.isna(ner_tagged_text) or ner_tagged_text is None:
        return []

    ner_tagged_text = str(ner_tagged_text)
    pattern = r'\[([A-Z_]+):\s*([^\]]+)\]'
    entities = []
    plain_index = 0
    cursor = 0

    for match in re.finditer(pattern, ner_tagged_text):
        entity_type = match.group(1)
        entity_text = match.group(2)
        prefix = ner_tagged_text[cursor:match.start()]
        plain_index += len(prefix)
        entities.append({
            'entity_type': entity_type,
            'entity_text': entity_text,
            'start_offset': plain_index,
            'end_offset': plain_index + len(entity_text),
        })
        plain_index += len(entity_text)
        cursor = match.end()

    return entities

In [4]:
# Compute ASR-side medical NER for each model (self-contained; ~3 x len(df) Healthcare API calls).
print("Running NER on ASR transcripts (whisper, phi4, parakeet)...")

df['norm_whisper_asr_ner'] = df['norm_whisper_asr'].apply(
    lambda t: apply_ner_to_row(t, PROJECT_ID, LOCATION)
)
df['norm_phi4_asr_ner'] = df['norm_phi4_asr'].apply(
    lambda t: apply_ner_to_row(t, PROJECT_ID, LOCATION)
)
df['norm_parakeet_asr_ner'] = df['norm_parakeet_asr'].apply(
    lambda t: apply_ner_to_row(t, PROJECT_ID, LOCATION)
)

print("✓ ASR NER complete: norm_whisper_asr_ner, norm_phi4_asr_ner, norm_parakeet_asr_ner")

Running NER on ASR transcripts (whisper, phi4, parakeet)...
Error calling healthcare API: timed out
✓ ASR NER complete: norm_whisper_asr_ner, norm_phi4_asr_ner, norm_parakeet_asr_ner


In [5]:
# Core comparison functions: exact, case-insensitive medical-entity-set comparison.

def build_medical_entity_set(ner_tagged_text):
    """Set of lowercased medical entity surface strings in an NER-tagged string."""
    return {e['entity_text'].strip().lower()
            for e in extract_medical_entities_from_ner_tags(ner_tagged_text)}


def retag_human_ner_against_asr(human_ner, asr_ner, missing_tag='MISS'):
    """Rewrite the human NER string: medical entities present in the ASR set are
    emitted as plain text (flow preserved); those absent become [MISS:span]."""
    if pd.isna(human_ner) or human_ner is None:
        return human_ner
    asr_set = build_medical_entity_set(asr_ner)
    tag_pattern = r'\[([A-Z_]+):\s*([^\]]+)\]'

    def _sub(m):
        span = m.group(2)
        return span if span.strip().lower() in asr_set else f"[{missing_tag}:{span}]"

    return re.sub(tag_pattern, _sub, str(human_ner))


def asr_extra_medical(human_ner, asr_ner):
    """ASR-only medical entities (hallucinations / mis-recognitions into medical)."""
    human_set = build_medical_entity_set(human_ner)
    seen, extras = set(), []
    for e in extract_medical_entities_from_ner_tags(asr_ner):
        key = e['entity_text'].strip().lower()
        if key not in human_set and key not in seen:
            seen.add(key)
            extras.append(e['entity_text'])
    return extras

In [6]:
# Apply the comparison per model, anchored on the human transcript NER.
models = ['whisper', 'phi4', 'parakeet']

for model in models:
    asr_ner_col = f'norm_{model}_asr_ner'

    df[f'{model}_reconstructed_ner_union'] = df.apply(
        lambda row: retag_human_ner_against_asr(
            row.get('norm_human_transcript_ner'),
            row.get(asr_ner_col),
        ),
        axis=1,
    )
    df[f'{model}_asr_extra_medical'] = df.apply(
        lambda row: asr_extra_medical(
            row.get('norm_human_transcript_ner'),
            row.get(asr_ner_col),
        ),
        axis=1,
    )
    df[f'{model}_n_missing'] = df[f'{model}_reconstructed_ner_union'].apply(
        lambda s: str(s).count('[MISS:')
    )
    df[f'{model}_n_extra'] = df[f'{model}_asr_extra_medical'].apply(len)

print("✓ Built per-model columns:")
for model in models:
    print(f"  {model}_reconstructed_ner_union, {model}_asr_extra_medical, "
          f"{model}_n_missing, {model}_n_extra")
print("\nMissing/extra medical-entity totals per model:")
for model in models:
    print(f"  {model}: missing={df[f'{model}_n_missing'].sum()}, "
          f"extra={df[f'{model}_n_extra'].sum()}")

✓ Built per-model columns:
  whisper_reconstructed_ner_union, whisper_asr_extra_medical, whisper_n_missing, whisper_n_extra
  phi4_reconstructed_ner_union, phi4_asr_extra_medical, phi4_n_missing, phi4_n_extra
  parakeet_reconstructed_ner_union, parakeet_asr_extra_medical, parakeet_n_missing, parakeet_n_extra

Missing/extra medical-entity totals per model:
  whisper: missing=1044, extra=955
  phi4: missing=1039, extra=893
  parakeet: missing=1014, extra=1007


In [7]:
# Save the experiment output and preview a few rows.
output_cols = ['utterance_id', 'source', 'norm_human_transcript', 'norm_human_transcript_ner']
for model in models:
    output_cols += [
        f'norm_{model}_asr_ner',
        f'{model}_reconstructed_ner_union',
        f'{model}_asr_extra_medical',
        f'{model}_n_missing',
        f'{model}_n_extra',
    ]
output_cols = [c for c in output_cols if c in df.columns]

out_df = df[output_cols].copy()
# Excel cannot store python lists; serialize the extra-medical lists to strings.
for model in models:
    col = f'{model}_asr_extra_medical'
    if col in out_df.columns:
        out_df[col] = out_df[col].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)

# out_df.to_excel('results/ner_union_experiment.xlsx', index=False, engine='openpyxl')
# print(f"✓ Saved results/ner_union_experiment.xlsx with {len(out_df)} rows")

# Preview: show the human flow with [MISS:..] tags for the whisper model.
for idx in range(min(3, len(df))):
    print(f"\nRow {idx} (utterance_id={df['utterance_id'].iloc[idx]}):")
    print("union:", str(df['whisper_reconstructed_ner_union'].iloc[idx])[:300])
    print("asr_extra_medical:", df['whisper_asr_extra_medical'].iloc[idx])


Row 0 (utterance_id=day5_consultation02):
union: good morning i am doctor smith from babylon can you just confirm your name date of birth and the first line of your address please hi my name is susan  thirty redbridge street sw two two hz hello and your date of birth forty oh two nineteen seventy four okay are you in a private place so you can hav
asr_extra_medical: ['thirty', 'fourteentwoone', 'spotted blood in my urine', 'loin pain', 'ibs', 'blood', 'meberine', 'sachets', 'passing urine']

Row 1 (utterance_id=day3_consultation02):
union:  hello hi i am doctor jacob and welcome to babylon hi  hi so just before we start is it alright if you could confirm your name for me please yep  john doe okay and your date of birth  uhh  twentyone twelve and nineteen  eightysix and your address for me please is  number one london street  nw three 
asr_extra_medical: ['jacob', 'twentyonetwelveone', 'nine hundred', 'eightysix', 'nwthirtysixpq', 'medical or surgical complaints', 'malaria', 'nonblanch